# DFU Repair-7 CPU LOCKED-PATH
CPU-only loader. Removes the real rr.build_manifest() call and rehydrates image paths only from the existing locked split relative_path values.

In [ ]:
import ast, json, urllib.request

V17_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/ba99c6b6da8ebd5a771e0eef6424586ee0933c02/notebooks/DFU_Repair7_v1_7_CPU_Colab.ipynb"

raw = urllib.request.urlopen(V17_URL, timeout=120).read()
nb = json.loads(raw.decode("utf-8"))
cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
if len(cells) != 1:
    raise RuntimeError(f"Expected exactly one v1.7 code cell, found {len(cells)}")
code = "".join(cells[0]["source"])

old_safety = '# Static safety assertions before execution.\nif script.count("rr.train_trial(") != 1:\n    raise RuntimeError(f"Safety check: expected exactly one rr.train_trial call, found {script.count(\'rr.train_trial(\')}")\nfor forbidden in ["rr.make_outer_folds(", "rr.assign_duplicate_groups(", "rr.build_manifest("]:\n    if forbidden in script:\n        raise RuntimeError(f"Safety check: forbidden split-regeneration call found: {forbidden}")\n'
new_safety = '# Replace the only real rr.build_manifest() usage with locked relative-path rehydration.\n_manifest_old = """dataset_root=rr.download_dataset(cfg,dirs)\ncurrent_manifest=rr.build_manifest(dataset_root,cfg,dirs)\ncurrent_manifest=current_manifest.loc[~current_manifest.exclude].copy()\npath_map=current_manifest[["image_id","image_path"]].drop_duplicates("image_id")\ndata=locked.drop(columns=["image_path"],errors="ignore").merge(path_map,on="image_id",how="left",validate="one_to_one")\nif data.image_path.isna().any():\n    raise RuntimeError(f"Current dataset is missing {int(data.image_path.isna().sum())} locked images; refusing repair.")\n"""\n_manifest_new = """dataset_root=rr.download_dataset(cfg,dirs)\ndata=locked.drop(columns=["image_path"],errors="ignore").copy()\ndata["image_path"]=[str((Path(dataset_root)/str(rp)).resolve()) for rp in data["relative_path"].astype(str)]\n_missing_paths=[p for p in data["image_path"].tolist() if not Path(p).is_file()]\nif _missing_paths:\n    raise RuntimeError(f"Current dataset is missing {len(_missing_paths)} locked image paths; refusing repair. First missing: {_missing_paths[0]}")\n"""\nif script.count(_manifest_old) != 1:\n    raise RuntimeError(f"Locked-path patch expected exactly one rr.build_manifest block, found {script.count(_manifest_old)}")\nscript = script.replace(_manifest_old, _manifest_new, 1)\nprint("Locked relative-path rehydration patch: PASS")\n\n# AST-based safety assertions: inspect REAL rr.* calls only.\n_tree = ast.parse(script)\n_rr_calls = []\nfor _node in ast.walk(_tree):\n    if (\n        isinstance(_node, ast.Call)\n        and isinstance(_node.func, ast.Attribute)\n        and isinstance(_node.func.value, ast.Name)\n        and _node.func.value.id == "rr"\n    ):\n        _rr_calls.append(_node.func.attr)\n\nif _rr_calls.count("train_trial") != 1:\n    raise RuntimeError(\n        f"AST safety check: expected exactly one real rr.train_trial() call, found {_rr_calls.count(\'train_trial\')}"\n    )\n\n_forbidden = {"make_outer_folds", "assign_duplicate_groups", "build_manifest"}\n_bad = sorted(_forbidden.intersection(_rr_calls))\nif _bad:\n    raise RuntimeError(f"AST safety check: forbidden real rr.* call(s) found after patch: {_bad}")\n\nprint("AST safety check: PASS — no split-regeneration calls")\n'

if code.count(old_safety) != 1:
    raise RuntimeError(f"Patch target expected once, found {code.count(old_safety)}")
code = code.replace(old_safety, new_safety, 1)
code = code.replace("DFU Repair-7 v1.7 CPU ONLY", "DFU Repair-7 CPU LOCKED-PATH", 1)
code = code.replace(
    "Pinned Repair-7 v1.7 CPU patch verification: PASS",
    "Pinned Repair-7 CPU LOCKED-PATH verification: PASS",
    1,
)

compile(code, "DFU_Repair7_CPU_LOCKED_PATH.py", "exec")

print("DFU Repair-7 CPU LOCKED-PATH loader: PASS")
print("CPU mode: ON")
print("38 good trials: READ-ONLY")
print("Authorized retraining: EXACT 7")
print("Dataset split source: EXISTING locked_outer_fold_assignments.csv ONLY")
exec(compile(code, "DFU_Repair7_CPU_LOCKED_PATH.py", "exec"), globals())
